# 全部链Rg
读取基础Rg结果，不筛选浓相；浓相和中心链Rg对比见分子内Notebook。先帧内链平均，再对帧等权；误差是时间块SEM。


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"simulations.yaml").is_file())
sys.path.insert(0,str(ROOT))
from common import inputs
from common import results as files
SELECTED=None
RESULTS_ROOT=ROOT/"03_results"
BLOCK_FRAMES=50
FIGURES=ROOT/"05_plot/01_basic/figures"
FIGURES.mkdir(parents=True,exist_ok=True)
fig,axes=plt.subplots(1,2,figsize=(10,3.5),constrained_layout=True)
sources={}
for x,sim in enumerate(inputs.select_simulations(ROOT/"simulations.yaml",SELECTED)):
    name=sim["id"];path=files.completed(RESULTS_ROOT/name/"01_basic/rg")
    sources[name]=files.read_json(path/"metadata.json")
    data=pd.read_csv(path/"chain_rg.csv.gz").groupby("frame").rg_nm.mean()
    blocks=data.groupby(data.index//BLOCK_FRAMES).mean()
    sem=blocks.std()/np.sqrt(blocks.count())
    axes[0].errorbar(x,data.mean(),yerr=sem if np.isfinite(sem) else None,fmt="o",capsize=4)
    axes[1].plot(data.index,data,label=name,lw=.6)
axes[0].set(xticks=range(len(sources)),xticklabels=list(sources),ylabel="All-chain mean Rg (nm)")
axes[0].tick_params(axis="x",labelrotation=20)
axes[1].set(xlabel="Analysis frame",ylabel="All-chain mean Rg (nm)");axes[1].legend()
for name,saved in sources.items():
    if files.read_json(files.completed(RESULTS_ROOT/name/"01_basic/rg")/"metadata.json")!=saved:raise ValueError("Results changed; rerun notebook")
for ext in ("png","pdf","svg"):fig.savefig(FIGURES/f"rg.{ext}",dpi=300,bbox_inches="tight")
files.write_json(FIGURES/"rg.sources.json",sources)
plt.show()
